# Прогноз остатков с CatBoost

Отдельная модель на каждый горизонт от 1 до 12 месяцев. На входе `df` с `REP_DT`, `OST_VAL` и признаками, известными на дату `REP_DT`. Если есть `CURENCY` или `CURRENCY`, строки каждого ряда связываются по своей валюте, а метрики считаются по валютам.

Если пакетов ещё нет, выполни в отдельной ячейке `%pip install catboost "mlflow>=3,<4" "pandas>=2" matplotlib`.

## 1. Данные и настройки

In [ ]:
import re

import matplotlib.pyplot as plt
import mlflow
import mlflow.catboost
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool, __version__ as catboost_version

# Здесь загрузи свои данные в df. Пример для проверки на файле проекта:
# df = pd.read_csv("df_fl_synthetic.csv").drop_duplicates(["REP_DT", "CURENCY"])

HORIZONS = range(1, 13)
N_FOLDS = 3
ITERATIONS = 400
TEST_START = None  # Или дата выпуска первого тестового прогноза, например "2024-01-01".
FEATURES = [col for col in df.columns if col != "REP_DT"]  # Можно указать список вручную.

In [ ]:
currency = next((col for col in ("CURENCY", "CURRENCY") if col in df), None)
series = [currency] if currency else []
data = df[list(dict.fromkeys(["REP_DT", "OST_VAL", *series, *FEATURES]))].copy()
dates = pd.to_datetime(data["REP_DT"], format="mixed", errors="coerce")
data["REP_DT"] = dates.dt.tz_localize(None).dt.normalize()
data["OST_VAL"] = pd.to_numeric(data["OST_VAL"], errors="coerce").replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=["REP_DT", "OST_VAL"]).sort_values("REP_DT").reset_index(drop=True)

cat_cols = data[FEATURES].select_dtypes(exclude=["number", "bool"]).columns.tolist()
for col in set(cat_cols + series):
    data[col] = data[col].astype("string").fillna("__missing__")
if currency in FEATURES and currency not in cat_cols:
    cat_cols.append(currency)
num_cols = [col for col in FEATURES if col not in cat_cols]
data[num_cols] = data[num_cols].astype(float).replace([np.inf, -np.inf], np.nan)

if data.empty or data.duplicated(["REP_DT", *series]).any():
    raise ValueError("Нужна одна строка на дату для каждого ряда остатков.")
print(f"Подготовлено {len(data):,} из {len(df):,} строк")

## 2. Остаток через нужное число месяцев

Для дневных данных ищем факт на календарную дату `REP_DT + h месяцев`. Если в каждом ряду одна строка на месяц, ищем факт в целевом месяце. Входные признаки остаются с даты выпуска.

In [ ]:
month = data["REP_DT"].dt.to_period("M")
is_monthly = data.groupby([*series, month]).size().max() == 1

actual = data[["REP_DT", "OST_VAL", *series]].rename(
    columns={"REP_DT": "target_dt", "OST_VAL": "target"}
)
actual["date_key"] = month if is_monthly else actual["target_dt"]
actual = actual.set_index([*series, "date_key"])

def sample_for(h):
    date_key = month + h if is_monthly else data["REP_DT"] + pd.DateOffset(months=h)
    return (
        data.assign(date_key=date_key)
        .join(actual, on=[*series, "date_key"], how="inner", validate="many_to_one")
        .drop(columns="date_key")
        .sort_values("REP_DT")
    )

eligible = sample_for(max(HORIZONS))["REP_DT"].drop_duplicates()
if eligible.empty:
    raise ValueError(f"Нет известных фактов для горизонта {max(HORIZONS)} мес.")
test_start = pd.Timestamp(TEST_START or eligible.iloc[int(0.8 * len(eligible))])
test_start = test_start.tz_localize(None).normalize()
print(f"Первый выпуск test с {test_start.date()}")

## 3. Временные фолды и обучение

В каждом фолде обучаемся на фактах, известных до первого выпуска `valid`. Для `test` переобучаем модель на всех фактах, известных до `test_start`. По умолчанию граница теста оставляет около 20% дат выпуска с известным фактом для самого длинного горизонта.

`train` показывает результат на обучающих строках первого фолда. `valid` объединяет прогнозы всех фолдов на их валидационных строках. Базовый прогноз равен текущему остатку, MAPE считается по ненулевым фактам.

In [ ]:
params = dict(
    iterations=ITERATIONS, depth=6, learning_rate=0.05, loss_function="MAE",
    random_seed=42, has_time=True, verbose=False, allow_writing_files=False,
    allow_const_label=True, thread_count=4,
)

def fit(rows):
    return CatBoostRegressor(**params).fit(rows[FEATURES], rows["target"], cat_features=cat_cols)

def predict(model, rows, split, fold=None):
    return rows[["REP_DT", "target_dt", "target", *series]].assign(
        baseline=rows["OST_VAL"], forecast=model.predict(rows[FEATURES]), split=split, fold=fold
    )

def score(rows, by):
    error = (rows["target"] - rows["forecast"]).abs()
    baseline_error = (rows["target"] - rows["baseline"]).abs()
    denom = rows["target"].abs().replace(0, np.nan)
    return rows.assign(
        MAE=error, MAPE_pct=100 * error / denom,
        baseline_MAE=baseline_error, baseline_MAPE_pct=100 * baseline_error / denom,
    ).groupby(by, as_index=False).agg(
        MAE=("MAE", "mean"), MAPE_pct=("MAPE_pct", "mean"),
        baseline_MAE=("baseline_MAE", "mean"), baseline_MAPE_pct=("baseline_MAPE_pct", "mean"),
        n=("target", "size"), n_MAPE=("MAPE_pct", "count"),
    )

In [ ]:
if not mlflow.config.is_tracking_uri_set():
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("catboost-monthly-balances")
client = mlflow.MlflowClient()
signature = mlflow.models.infer_signature(data[FEATURES].head(), data["OST_VAL"].head().to_numpy(float))

models, run_ids, shap_sample = {}, {}, {}
predictions, metrics, cv_metrics = [], [], []

for h in HORIZONS:
    sample = sample_for(h)
    development = sample[sample["target_dt"] < test_start]
    test = sample[sample["REP_DT"] >= test_start]
    # Первому фолду нужны хотя бы два выпуска с уже известными фактами.
    first_facts = development.sort_values("target_dt").drop_duplicates("REP_DT")["target_dt"].head(2)
    valid_dates = development["REP_DT"].drop_duplicates()
    valid_dates = valid_dates.iloc[len(valid_dates) // 2:]
    valid_dates = valid_dates[valid_dates > first_facts.max()]
    if len(first_facts) < 2 or len(valid_dates) < N_FOLDS or test.empty:
        raise ValueError(f"Мало истории для {h} мес. Проверь TEST_START и доступные даты.")

    parts = []
    for fold, dates in enumerate(np.array_split(valid_dates.to_numpy(), N_FOLDS), start=1):
        train = development[development["target_dt"] < dates[0]]
        valid = development[development["REP_DT"].isin(dates)]
        model = fit(train)
        if fold == 1:
            parts.append(predict(model, train, "train", fold))
        parts.append(predict(model, valid, "valid", fold))

    models[h] = fit(development)
    parts.append(predict(models[h], test, "test"))
    rows = pd.concat(parts, ignore_index=True).assign(months=h)
    scores = score(rows, ["months", "split", *series])
    folds = score(rows[rows["split"] == "valid"], ["months", "fold", *series])
    predictions.append(rows)
    metrics.append(scores)
    cv_metrics.append(folds)
    shap_sample[h] = test[FEATURES].sample(n=min(len(test), 500), random_state=42)

    with mlflow.start_run(run_name=f"{h}m") as run:
        run_ids[h] = run.info.run_id
        mlflow.log_params({**params, "months": h, "test_start": str(test_start.date()), "n_folds": N_FOLDS})
        for row in scores.to_dict("records"):
            name = "/".join([row["split"], *(str(row[col]) for col in series)])
            name = re.sub(r"[^a-zA-Z0-9_./ -]", "_", name)
            mlflow.log_metric(f"{name}/MAE", row["MAE"])
            if pd.notna(row["MAPE_pct"]):
                mlflow.log_metric(f"{name}/MAPE_pct", row["MAPE_pct"])
        mlflow.log_text(scores.to_csv(index=False), "metrics.csv")
        mlflow.log_text(folds.to_csv(index=False), "cv_metrics.csv")
        mlflow.catboost.log_model(
            models[h], name="catboost", signature=signature,
            pip_requirements=[f"catboost=={catboost_version}", f"pandas=={pd.__version__}"],
        )

predictions = pd.concat(predictions, ignore_index=True)
metrics = pd.concat(metrics, ignore_index=True)
cv_metrics = pd.concat(cv_metrics, ignore_index=True)

## 4. Качество на test и по фолдам

Для локального MLflow запусти `mlflow ui --backend-store-uri sqlite:///mlflow.db` в каталоге ноутбука.

In [ ]:
display(metrics.query("split == 'test'"))
display(cv_metrics)

## 5. Факт и прогноз

In [ ]:
PLOT_MONTHS = 3 if 3 in models else next(iter(models))  # Любой обученный горизонт.
chosen = predictions[predictions["months"] == PLOT_MONTHS]
PLOT_CURRENCY = chosen[currency].iloc[0] if currency and not chosen.empty else None  # Например, "RUB".
history = data
if currency:
    chosen = chosen[chosen[currency] == PLOT_CURRENCY]
    history = history[history[currency] == PLOT_CURRENCY]
if chosen.empty:
    raise ValueError("Для выбранного горизонта и валюты нет прогнозов.")

fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
ax.plot(history["REP_DT"], history["OST_VAL"], color="0.35", linewidth=1, label="Факт")
for split, part in chosen.sort_values("target_dt").groupby("split", sort=False):
    ax.plot(part["target_dt"], part["forecast"], marker=".", markersize=3, label=split)

plot_title = f"Горизонт {PLOT_MONTHS} мес."
ax.set(title=f"{plot_title} · {PLOT_CURRENCY}" if currency else plot_title,
       xlabel="Дата целевого остатка", ylabel="OST_VAL")
ax.legend()
ax.grid(alpha=0.2)
client.log_figure(run_ids[PLOT_MONTHS], fig, "forecast.png")
plt.show()

## 6. SHAP

`shap_importance` содержит средний абсолютный вклад каждого признака на тестовых строках всех моделей. На графике показано десять крупнейших вкладов для выбранного горизонта.

In [ ]:
importance = []
for h, model in models.items():
    pool = Pool(shap_sample[h], cat_features=cat_cols)
    values = model.get_feature_importance(pool, type="ShapValues", thread_count=4)
    # Последний столбец CatBoost содержит базовое значение прогноза.
    table = pd.DataFrame({
        "months": h, "feature": FEATURES, "mean_abs_shap": np.abs(values[:, :-1]).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False)
    importance.append(table)
    client.log_text(run_ids[h], table.to_csv(index=False), "shap.csv")

shap_importance = pd.concat(importance, ignore_index=True)
top = shap_importance.query("months == @PLOT_MONTHS").head(10).set_index("feature")
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
top["mean_abs_shap"].sort_values().plot.barh(ax=ax, title=f"SHAP · {PLOT_MONTHS} мес.")
ax.set(xlabel="Среднее |SHAP|", ylabel="")
client.log_figure(run_ids[PLOT_MONTHS], fig, "shap.png")
plt.show()